# SIMPLER (real-to-sim): Octo and OpenVLA, clean vs camera bump vs lighting
**GPU: L4** (needs Vulkan; A100 also works). WidowX visual-matching tasks, whose success tracks the real robot.
5 conditions x 4 tasks x 24 episodes per policy. Octo ~1 h, OpenVLA ~2 h.
The smoke cell checks Vulkan rendering first; if it fails, stop and paste the error.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline matplotlib backend crashes every venv python
os.environ['PYTHONUNBUFFERED'] = '1'   # run cells are piped into a log on Drive; without this their output arrives in bursts
os.makedirs('/content/drive/MyDrive/restoring_organs/plus/logs', exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)
print('outputs ->', '/content/drive/MyDrive/restoring_organs/plus')
# Drive uploads a file only after it is CLOSED, so a log held open by a running cell never arrives. Run cells log to
# /content/logs; this thread rewrites the last 64 KB of each log on Drive once a minute as a fresh, closed file.
import builtins, glob, threading, time
def _mirror_logs():
    while True:
        for p in glob.glob('/content/logs/*.log'):
            try:
                with open(p, 'rb') as f:
                    f.seek(max(0, os.path.getsize(p) - 65536))
                    tail = f.read()
                with open('/content/drive/MyDrive/restoring_organs/plus/logs/' + os.path.basename(p)[:-4] + '.' + os.uname().nodename[-6:] + '.log', 'wb') as f:
                    f.write(tail)
            except Exception:
                pass
        time.sleep(60)
if not getattr(builtins, '_log_mirror_on', False):
    builtins._log_mirror_on = True
    threading.Thread(target=_mirror_logs, daemon=True).start()
# Hugging Face login from the Colab secret HF_TOKEN (key icon, left sidebar), if you added one; optional here
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('Hugging Face login ok')
except Exception as e:
    print('no HF_TOKEN secret, continuing without Hugging Face login:', type(e).__name__)

In [ ]:
import os; os.makedirs('/content/scripts', exist_ok=True); print('ok')

In [ ]:
%%writefile /content/scripts/plus_common.py
"""Shared by every LIBERO-Plus notebook: one deterministic task subset (so all five policy families see
exactly the same perturbed tasks), perturbation categories parsed from task names, and the summary table."""
import json
import math
import os
import random
import re
from collections import Counter, defaultdict

# LIBERO-Plus task names, checked on the real suite: token perturbations carry their own tag; camera and robot
# initial state are encoded as values, e.g. ..._view_0_0_100_0_0_initstate_0 is the unperturbed setting.
CATS = [("sensor_noise", r"_noise_\d+"), ("language", r"_language_\d+"), ("light", r"_light_\d+"),
        ("background", r"_table_\d+|_tb_\d+"), ("layout", r"_add_\d+|_level")]
DEFAULT_VIEW = ("0", "0", "100", "0", "0")
# Every run script reads the suite from the environment, so one variable switches a whole notebook:
#   LIBERO_SUITE=libero_spatial | libero_object | libero_goal | libero_10 (default; nothing changes when it is unset)
SUITE = os.environ.get("LIBERO_SUITE", "libero_10")
MAX_STEPS = {"libero_spatial": 220, "libero_object": 280, "libero_goal": 300, "libero_10": 520}     # OFT's own horizons
OFT_CKPT = {s: "moojink/openvla-7b-oft-finetuned-" + s.replace("_", "-") for s in MAX_STEPS}


def categorize(name):
    for cat, pat in CATS:
        if re.search(pat, name):
            return cat
    m = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", name)
    if m and m.groups() != DEFAULT_VIEW:
        return "camera"
    s = re.search(r"_initstate_(\d+)", name)
    if s and s.group(1) != "0":
        return "robot_init"
    return "other"


def build_subset(suite_name="libero_10", per_cat=30, seed=0):
    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()[suite_name]()
    names = [suite.get_task(i).name for i in range(suite.n_tasks)]
    langs = [suite.get_task(i).language for i in range(suite.n_tasks)]
    by = defaultdict(list)
    for i, n in enumerate(names):
        by[categorize(n)].append(i)
    rng = random.Random(seed)
    pick = []
    for cat in sorted(by):
        ids = sorted(by[cat])
        rng.shuffle(ids)
        pick += ids[:per_cat]
    pick.sort()
    tokens = Counter(tok for n in names for tok in re.findall(r"_([a-z]+)_\d+", n))
    return {"suite": suite_name, "n_tasks": suite.n_tasks, "per_cat": per_cat, "seed": seed,
            "counts": {c: len(v) for c, v in sorted(by.items())},
            "other_examples": [names[i] for i in by.get("other", [])[:5]],
            "name_tokens": dict(tokens.most_common(20)),
            "tasks": [{"task_id": i, "name": names[i], "language": langs[i], "category": categorize(names[i])}
                      for i in pick]}


def load_or_build(path, per_cat=30):
    if os.path.exists(path):
        return json.load(open(path))
    sub = build_subset(suite_name=SUITE, per_cat=per_cat)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    json.dump(sub, open(path, "w"), indent=1)
    return sub


def load_rows(path):
    rows = []
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows


def append_row(path, row):
    """Append one result row and CLOSE the file. Drive uploads a file only when it is closed: a runner that keeps its
    results file open for a whole cell has nothing on Drive until the cell ends, and a killed runtime loses all of it."""
    with open(path, "a") as f:
        f.write(json.dumps(row, default=float) + "\n")


def load_shards(path):
    """Rows of `<stem>.jsonl` plus every per-session shard `<stem>.<shard>.jsonl` next to it, one row per
    (task, condition, arm, rep). Two Colab sessions that append to ONE Drive file overwrite each other (each uploads its
    own whole copy, the last upload wins; file 8 lost 576 rows that way on 2026-09-20), so every session writes its own
    shard and readers merge. A measured row beats one recovered from a log (`recovered`), which lacks the detail fields."""
    import glob
    best = {}
    # 0 = measured, 1 = measured before the shading gate (re-queued by the runner, its re-run lives in a later shard),
    # 2 = rebuilt from a log. The lowest rank wins; on a tie the first file does.
    rank = lambda r: 2 if r.get("recovered") else int(bool(r.get("photo_applied")) and "photo_shading" not in r)
    for p in [path] + sorted(glob.glob(path[:-len(".jsonl")] + ".*.jsonl")):
        for r in load_rows(p):
            k = (r.get("policy"), r["task_id"], r.get("cond"), r["arm"], r.get("rep", 0))
            if k not in best or rank(r) < rank(best[k]):
                best[k] = r
    return list(best.values())


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)


def summarize(rows, label=""):
    by = defaultdict(list)
    for r in rows:
        by[r["category"]].append(bool(r["success"]))
    out = {}
    print(f"\n{label}  ({len(rows)} episodes)")
    print(f"{'category':<14}{'n':>5}{'success':>9}{'95% CI':>16}")
    for cat in sorted(by) + ["ALL"]:
        v = [s for r in by.values() for s in r] if cat == "ALL" else by[cat]
        k, n = sum(v), len(v)
        lo, hi = wilson(k, n)
        out[cat] = {"n": n, "success": k / max(1, n), "ci": [lo, hi]}
        print(f"{cat:<14}{n:>5}{k / max(1, n):>9.3f}   [{lo:.2f}, {hi:.2f}]")
    return out

In [ ]:
%%writefile /content/scripts/simpler_eval.py
"""SIMPLER (real-to-sim, WidowX visual-matching envs): Octo and OpenVLA, clean and under camera and lighting
perturbations applied to the camera image. Loop mirrors simple_inference_visual_matching_prepackaged_envs.py."""
import argparse
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_rows, wilson  # noqa: E402

TASKS = ["widowx_spoon_on_towel", "widowx_carrot_on_plate", "widowx_stack_cube", "widowx_put_eggplant_in_basket"]
CONDS = ["clean", "cam3", "cam6", "dim", "bright"]


def perturb(img, cond):
    import cv2
    if cond == "clean":
        return img
    if cond.startswith("cam"):              # camera bump: rotation about the optical centre, as a homography
        deg = float(cond[3:])
        h, w = img.shape[:2]
        f = float(w)
        K = np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1]])
        ay, ax = np.deg2rad(deg), np.deg2rad(deg / 2)
        Ry = np.array([[np.cos(ay), 0, np.sin(ay)], [0, 1, 0], [-np.sin(ay), 0, np.cos(ay)]])
        Rx = np.array([[1, 0, 0], [0, np.cos(ax), -np.sin(ax)], [0, np.sin(ax), np.cos(ax)]])
        H = K @ Ry @ Rx @ np.linalg.inv(K)
        return cv2.warpPerspective(img, H, (w, h), borderMode=cv2.BORDER_REPLICATE)
    gain, gamma = {"dim": (0.55, 1.2), "bright": (1.45, 0.85)}[cond]
    x = np.clip((img.astype(np.float32) / 255.0) ** gamma * gain, 0, 1)
    return (x * 255).astype(np.uint8)


def make_model(policy):
    if policy.startswith("octo"):
        import tensorflow as tf
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=3072)])
        from simpler_env.policies.octo.octo_model import OctoInference
        return OctoInference(model_type=policy, policy_setup="widowx_bridge", init_rng=0)
    from simpler_env.policies.openvla.openvla_model import OpenVLAInference
    return OpenVLAInference(saved_model_path="openvla/openvla-7b", policy_setup="widowx_bridge")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--policy", required=True, help="octo-small | octo-base | openvla")
    ap.add_argument("--out", required=True)
    ap.add_argument("--n", type=int, default=24)
    ap.add_argument("--conds", default=",".join(CONDS))
    ap.add_argument("--tasks", default=",".join(TASKS))
    a = ap.parse_args()

    import simpler_env
    from simpler_env.utils.env.observation_utils import get_image_from_maniskill2_obs_dict
    model = make_model(a.policy)
    done = {(r["task"], r["cond"], r["ep"]) for r in load_rows(a.out)}
    with open(a.out, "a") as f:
        for task in a.tasks.split(","):
            env = simpler_env.make(task)
            for cond in a.conds.split(","):
                for ep in range(a.n):
                    if (task, cond, ep) in done:
                        continue
                    obs, _ = env.reset()
                    instruction = env.get_language_instruction()
                    is_final = env.is_final_subtask()
                    model.reset(instruction)
                    image = perturb(get_image_from_maniskill2_obs_dict(env, obs), cond)
                    terminated, success, truncated = False, False, False
                    while not (terminated or truncated):
                        _, action = model.step(image, instruction)
                        terminated = bool(action["terminate_episode"][0] > 0)
                        if terminated and not is_final:
                            terminated = False
                            env.advance_to_next_subtask()
                        obs, _, success, truncated, _ = env.step(
                            np.concatenate([action["world_vector"], action["rot_axangle"], action["gripper"]]))
                        new = env.get_language_instruction()
                        if new != instruction:
                            instruction = new
                        is_final = env.is_final_subtask()
                        image = perturb(get_image_from_maniskill2_obs_dict(env, obs), cond)
                    f.write(json.dumps(dict(policy=a.policy, task=task, cond=cond, ep=ep, success=bool(success))) + "\n")
                    f.flush()
                    print(f"{task} {cond} ep{ep} success={bool(success)}", flush=True)
            env.close()
    rows = load_rows(a.out)
    print(f"\n{a.policy}")
    for cond in CONDS:
        v = [r["success"] for r in rows if r["cond"] == cond]
        if v:
            lo, hi = wilson(sum(v), len(v))
            print(f"  {cond:<7} {sum(v)}/{len(v)} = {sum(v)/len(v):.3f}  [{lo:.2f}, {hi:.2f}]")


if __name__ == "__main__":
    main()

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y libvulkan1 vulkan-tools libglvnd-dev > /dev/null
mkdir -p /usr/share/vulkan/icd.d
cat > /usr/share/vulkan/icd.d/nvidia_icd.json <<'EOF'
{"file_format_version": "1.0.0", "ICD": {"library_path": "libGLX_nvidia.so.0", "api_version": "1.2.155"}}
EOF
cd /content
[ -d SimplerEnv-OpenVLA ] || git clone -q --recurse-submodules --depth 1 https://github.com/DelinQu/SimplerEnv-OpenVLA.git
pip -q install uv
[ -x /content/venv_simpler_octo/bin/python ] || uv venv -q /content/venv_simpler_octo -p 3.10
[ -x /content/venv_simpler_vla/bin/python ] || uv venv -q /content/venv_simpler_vla -p 3.10

uv pip install -q --python /content/venv_simpler_octo/bin/python numpy==1.24.4
uv pip install -q --python /content/venv_simpler_octo/bin/python -e /content/SimplerEnv-OpenVLA/ManiSkill2_real2sim
uv pip install -q --python /content/venv_simpler_octo/bin/python -e /content/SimplerEnv-OpenVLA
uv pip install -q --python /content/venv_simpler_octo/bin/python opencv-python-headless transforms3d matplotlib mediapy

uv pip install -q --python /content/venv_simpler_octo/bin/python tensorflow==2.15.0 "git+https://github.com/octo-models/octo.git"
uv pip install -q --python /content/venv_simpler_octo/bin/python "jax[cuda11_pip]==0.4.20" --find-links https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
uv pip install -q --python /content/venv_simpler_octo/bin/python "protobuf==4.25.3"
uv pip install -q --python /content/venv_simpler_octo/bin/python --no-deps "tensorflow-metadata==1.16.1"

uv pip install -q --python /content/venv_simpler_vla/bin/python numpy==1.24.4
uv pip install -q --python /content/venv_simpler_vla/bin/python -e /content/SimplerEnv-OpenVLA/ManiSkill2_real2sim
uv pip install -q --python /content/venv_simpler_vla/bin/python -e /content/SimplerEnv-OpenVLA
uv pip install -q --python /content/venv_simpler_vla/bin/python opencv-python-headless transforms3d matplotlib mediapy

uv pip install -q --python /content/venv_simpler_vla/bin/python torch==2.2.0 torchvision==0.17.0 transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate
sed -i 's/attn_implementation="flash_attention_2"/attn_implementation="sdpa"/' /content/SimplerEnv-OpenVLA/simpler_env/policies/openvla/openvla_model.py || true
echo done

In [ ]:
# smoke: Vulkan rendering + env reset in both venvs
!/content/venv_simpler_octo/bin/python -c "import simpler_env; from simpler_env.utils.env.observation_utils import get_image_from_maniskill2_obs_dict as g; e = simpler_env.make('widowx_spoon_on_towel'); o, _ = e.reset(); print('octo venv render', g(e, o).shape)"
!/content/venv_simpler_vla/bin/python -c "import simpler_env; from simpler_env.utils.env.observation_utils import get_image_from_maniskill2_obs_dict as g; e = simpler_env.make('widowx_spoon_on_towel'); o, _ = e.reset(); print('openvla venv render', g(e, o).shape)"

In [ ]:
# Octo (octo-small): full grid, resumable
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_simpler_octo/bin/python /content/scripts/simpler_eval.py --policy octo-small --out /content/drive/MyDrive/restoring_organs/plus/simpler_octo.jsonl 2>&1 | tee -a /content/logs/simpler_eval.log

In [ ]:
# OpenVLA: full grid, resumable
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; /content/venv_simpler_vla/bin/python /content/scripts/simpler_eval.py --policy openvla --out /content/drive/MyDrive/restoring_organs/plus/simpler_openvla.jsonl 2>&1 | tee -a /content/logs/simpler_eval.log